In [258]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LassoCV


In [259]:
def extract_series(df):
    df_temp = df

    # Convert 'id' column to string type
    df_temp['id'] = df_temp['id'].astype(str)
    # Add a new column 'series' which is the first three digits of 'id'
    df_temp['series'] = df_temp['id'].str[:4].astype(int)

    # Read the series.csv file
    series = pd.read_csv('Series Name.csv')

    # Merge data with series on the 'series' column from data and 'series_id' column from series
    df_temp_1 = df_temp.merge(series, left_on='series', right_on='Series Name ID', how='left')

    cols = df_temp_1.columns.tolist()
    cols.insert(1, cols.pop(cols.index('Series Name')))
    df_temp_1 = df_temp_1[cols]
    
    df_temp_1.drop(columns=['Series Name ID', 'series'], inplace=True)

    return df_temp_1

In [260]:
def extract_contry(df):
    df_temp = df
    # Add a new column 'country' which is the fourth and fifth digits of 'id'
    df_temp['country'] = df_temp['id'].str[4:6].astype(int)

# Read the country.csv file
    country = pd.read_csv('Country Name.csv')

# Merge data_series with country on the 'country' column from data_series and 'country_id' column from country
    df_temp_1 = df_temp.merge(country, left_on='country', right_on='Country Name ID', how='left')

# Insert 'location' column into the correct position
    cols = df_temp_1.columns.tolist()
    cols.insert(2, cols.pop(cols.index('Country Name')))
    df_temp_1 = df_temp_1[cols]

# Drop unnecessary columns
    df_temp_1.drop(columns=['Country Name ID', 'country'], inplace=True)
    
    return df_temp_1

In [261]:
def extract_category(df):
    df_temp = df
    # Add a new column 'country' which is the fourth and fifth digits of 'id'
    df_temp['category'] = df_temp['id'].str[6:8].astype(int)

# Read the country.csv file
    category = pd.read_excel('category_id.xlsx')

# Rename the 'id' column in the country dataframe to 'country_id'
    category.rename(columns={'id': 'category_id'}, inplace=True)

# Merge data_series with country on the 'country' column from data_series and 'country_id' column from country
    df_temp_1 = df_temp.merge(category, left_on='category', right_on='category_id', how='left')

# Insert 'location' column into the correct position
    cols = df_temp_1.columns.tolist()
    cols.insert(3, cols.pop(cols.index('Category')))
    df_temp_1 = df_temp_1[cols]

# Drop unnecessary columns
    df_temp_1.drop(columns=['category_id', 'category'], inplace=True)

    return df_temp_1

In [262]:
prosperity = pd.read_csv('prosperity.csv')
prosperity.columns = ['year', 'prosperity']

In [263]:
data = pd.read_csv('cleaned_data.csv')
data['id'] = data['id'].astype(str)
data['id'] = data['id'].apply(lambda x: f'{int(x):014}' if pd.notnull(x) else x)
#nf_data = data[data['id'].str[5:7] != '05']
data = data.set_index('id')
data_stand = data.copy()

In [265]:
for col in range(len(data)):
    data_stand.iloc[col] = (data.iloc[col] - data.iloc[col].min()) / (data.iloc[col].max() - data.iloc[col].min())
data_stand = data_stand.dropna()

In [266]:
len(data_stand.columns)

31

In [268]:
# Create a dataframe to store the id and p-values
pvalues_df = pd.DataFrame(columns=['id', 'pvalue'])

for col in range(len(data_stand)):

    # Ensure y and x have the same length
    x = data_stand.iloc[col].values.reshape(-1, 1)
    y = prosperity['prosperity']

    # Add a constant to the model (intercept)
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit()

    pvalues_df.loc[col, 'id'] = data.index[col]
    
    # Check if model.pvalues has an index 1
    if len(model.pvalues) > 1:
        pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
    else:
        pvalues_df.loc[col, 'pvalue'] = 1 # Assign NaN if p-value is not available


/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/2330189477.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/2330189477.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pvalues_df.loc[col, 'pvalue'] = model.pvalues[1]  # Extract the p-value for the predictor variable
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/2330189477.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a futu

In [269]:
pvalues_threshold = 0.01 / len(pvalues_df)
data_significant = pvalues_df[pvalues_df['pvalue'] < pvalues_threshold]

In [270]:
# Convert 'pvalue' column to numeric
pvalues_df['pvalue'] = pd.to_numeric(pvalues_df['pvalue'], errors='coerce')


In [271]:
temp = data_stand.reset_index()

# Get the top 10 rows with the smallest p-values
#nf_data_top10_pvalues = pvalues_df.nsmallest(100, 'pvalue')
#nf_data_top10_pvalues = nf_data_significant

data_significant_db = temp[temp['id'].isin(data_significant['id'])]

data_db = data_significant_db

In [272]:
data_significant_seires = extract_series(data_significant_db)
data_significant_country = extract_contry(data_significant_seires)
data_significant_category = extract_category(data_significant_country)

/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/395400177.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['id'] = df_temp['id'].astype(str)
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/395400177.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['series'] = df_temp['id'].str[:4].astype(int)


In [273]:
len(data_significant_category)

549

In [274]:
data_significant_category.to_excel('pvalue_data.xlsx')

In [275]:
data_significant_category.groupby('Series Name').count()['id'].sort_values(ascending=False).to_csv('significant.csv')

In [280]:
data_db = data_db.drop('series', axis=1)

In [281]:
data_significant_db_transposed = data_db.transpose().reset_index()
data_significant_db_transposed.columns = data_significant_db_transposed.iloc[0]
data_significant_clean = data_significant_db_transposed.drop(0)

data_significant_clean.set_index('id', inplace=True)
data_significant_clean.index.name = None


In [306]:
from collections import defaultdict

# Dictionary to store the distribution of nonzero coefficients for each id
nonzero_coefficients_distribution = defaultdict(list)

# Number of simulations
num_simulations = 10

for _ in range(num_simulations):
    # Shuffle the columns of data_significant_clean
    data_significant_clean_ran = data_significant_clean.sample(frac=1, axis=1, random_state=42)

    # Prepare X and y
    X = data_significant_clean_ran.astype(float)
    y = prosperity['prosperity']

    # Perform LassoCV to find the best alpha
    lasso_cv = LassoCV(cv=5, random_state=42)
    lasso_cv.fit(X, y)

    best_alpha = lasso_cv.alpha_

    # Fit the Lasso regression model
    lasso = Lasso(alpha=best_alpha, random_state=42)
    lasso.fit(X, y)
    # Get the nonzero coefficients and their corresponding ids
    nonzero_ids = X.columns[lasso.coef_ != 0]

    # Count the number of times each id has a nonzero coefficient
    for id_ in nonzero_ids:
        nonzero_coefficients_distribution[id_].append(1)

# Convert the distribution dictionary to a DataFrame for better visualization
nonzero_coefficients_df = pd.DataFrame(dict(nonzero_coefficients_distribution)).transpose()
nonzero_coefficients_df.columns = [f'Simulation_{i+1}' for i in range(num_simulations)]
nonzero_coefficients_df.index.name = 'id'

# Sum across simulations to get the total count of nonzero coefficients for each id
nonzero_coefficients_df['Total_Count'] = nonzero_coefficients_df.sum(axis=1)

print(nonzero_coefficients_df)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.262e-04, tolerance: 2.621e-04
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.262e-04, tolerance: 2.621e-04
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

                Simulation_1  Simulation_2  Simulation_3  Simulation_4  \
id                                                                       
12750102001275             1             1             1             1   
06690100000669             1             1             1             1   
05380104000538             1             1             1             1   
11060100001097             1             1             1             1   
04320104000432             1             1             1             1   
07910103000791             1             1             1             1   
02510100000231             1             1             1             1   
04710104000471             1             1             1             1   
03980105000399             1             1             1             1   
10650100001056             1             1             1             1   
04040105000404             1             1             1             1   
07730100000773             1          

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.262e-04, tolerance: 2.621e-04
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.262e-04, tolerance: 2.621e-04
  model = cd_fast.enet_coordinate_descent(


In [302]:
# Extract the Lasso coefficients and their corresponding feature indices
lasso_coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lasso.coef_
})

# Filter out features with zero coefficients
final_coefficients_df = lasso_coefficients[lasso_coefficients['Coefficient'] != 0].reset_index(drop=True)

final_coefficients_df.to_csv('lasso.csv')

In [303]:
temp = data_stand.reset_index()
lasso_db = temp[temp['id'].isin(final_coefficients_df['Feature'])]
lasso_seires = extract_series(lasso_db)
lasso_country = extract_contry(lasso_seires)
lasso_category = extract_category(lasso_country)

/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/395400177.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['id'] = df_temp['id'].astype(str)
/var/folders/qz/xwx1r5sx2k9dgf6nxmgph3200000gn/T/ipykernel_14497/395400177.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['series'] = df_temp['id'].str[:4].astype(int)


In [304]:
lasso_category.to_excel('lasso_data.xlsx')
lasso_category.groupby('Series Name').count()['id'].sort_values(ascending=False).to_csv('lasso_series.csv')
lasso_category.groupby('Country Name').count()['id'].to_csv('lasso_country.csv')